# Step 2 — Knowledge Wiki (Multimodal Qdrant, BM25-blend Retrieval)

Indexes **all six doc types** into Qdrant (transcript, financial, segment, geo, filing, news)
and demonstrates the **BM25-blend reranker** that works even with the offline hash embedder
(`docs/wiki-ingestion.md`).

Transport is configurable via `.env`:
- `QDRANT_MODE=memory` — in-process, zero setup (default)
- `QDRANT_MODE=docker` — `docker run -p 6333:6333 qdrant/qdrant`
- `QDRANT_MODE=local`  — on-disk at `QDRANT_PATH`

Embeddings: `EMBEDDING_BACKEND=hash` (offline) → `sentence-transformers` → `vllm` (MI300X).
The BM25 weight automatically adjusts per backend (0.65 offline → 0.15 on MI300X).

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root(); sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print(f"ticker={settings.ticker}  period={settings.period}")
print(f"data={'mock(real cached)' if settings.use_mock_data else 'live defeatbeta-api'}  "
      f"embeddings={settings.embedding_backend}  sentiment={settings.sentiment_backend}  llm={settings.llm_backend}")

ticker=NVDA  period=FY2026Q2
data=mock(real cached)  embeddings=hash  sentiment=lexicon  llm=mock


## Ingest all doc types

In [2]:
from ir_copilot.embeddings import get_embedder
from ir_copilot.vectorstore import WikiStore, WikiChunk, BM25Reranker
from ir_copilot.corpus import wiki_chunks_for
import collections

embedder = get_embedder()
print(f"embedder: {type(embedder).__name__}  dim={embedder.dim}  bm25_alpha={BM25Reranker().alpha:.2f}")

wiki = WikiStore(embedder)
wiki.ensure_collection(recreate=True)

# All six doc types for the company + peers
tickers = [settings.ticker, *settings.peers]
all_chunks = wiki_chunks_for(tickers)
by_type = collections.Counter(c["doc_type"] for c in all_chunks)
print(f"\nIngesting {len(all_chunks)} chunks across {len(tickers)} tickers:")
for dt, cnt in sorted(by_type.items()): print(f"  {dt:12} {cnt}")

chunk_objs = [WikiChunk(chunk_id=str(i), **c) for i, c in enumerate(all_chunks)]
n = wiki.upsert(chunk_objs)
print(f"\nQdrant transport: {wiki.mode}  |  upserted {n} chunks into '{wiki.collection}'")

embedder: HashEmbedder  dim=384  bm25_alpha=0.65



Ingesting 177 chunks across 3 tickers:
  filing       36
  financial    3
  geo          2
  news         24
  segment      4
  transcript   108

Qdrant transport: memory  |  upserted 177 chunks into 'ir_wiki'


## Retrieval: typed search + cross-type BM25-blend

`search(doc_type=None)` fetches 4× k from Qdrant then reranks by the BM25 blend so diverse
doc types surface.  `search_each_type()` guarantees coverage across all six types.

In [3]:
queries = [
    ("revenue by segment data center compute networking",  None),
    ("gross margin operating margin profitability",        None),
    ("SEC 10-K annual report filing",                      "filing"),
    ("analyst question guidance outlook next quarter",     "transcript"),
    ("news AI competition market share",                   "news"),
]

for q, dt in queries:
    label = f"[{dt or 'ALL TYPES'}]"
    print(f"\n── {label} '{q[:50]}'")
    hits = wiki.search(q, ticker=settings.ticker, k=3, doc_type=dt)
    for h in hits:
        print(f"  [{h['score']:.3f} v={h.get('vector_score','?'):.3f} "
              f"bm25={h.get('bm25_score','?'):.3f}] ({h['doc_type']:10}) {h['text'][:70]}...")
        print(f"    src: {h['source_url'][:80]}")


── [ALL TYPES] 'revenue by segment data center compute networking'
  [0.822 v=0.491 bm25=1.000] (segment   ) NVDA Revenue by Segment (2026-04-26): Compute and Networking revenue $...
    src: https://huggingface.co/datasets/defeatbeta/yahoo-finance-data (quarterly_revenue
  [0.457 v=0.415 bm25=0.480] (segment   ) NVDA Revenue by Market Platform (2026-04-26): Data Center Revenue $75....
    src: https://huggingface.co/datasets/defeatbeta/yahoo-finance-data (quarterly_revenue
  [0.368 v=0.333 bm25=0.387] (transcript) We delivered another record quarter while navigating what continues to...
    src: https://huggingface.co/datasets/defeatbeta/yahoo-finance-data (earning_call_tran

── [ALL TYPES] 'gross margin operating margin profitability'
  [0.056 v=0.160 bm25=0.000] (news      ) Why SSR Mining Stock Got Knocked Today — Gold and silver are eternally...
    src: https://finance.yahoo.com/m/74a09df1-8fb5-3922-91c6-637dc75bbce6/why-ssr-mining-
  [0.056 v=0.159 bm25=0.000] (transcript) Yes.

In [4]:
# Cross-type balanced: guaranteed one hit per doc type
print("── search_each_type (guaranteed doc-type coverage):")
for h in wiki.search_each_type("revenue margin performance competition", ticker=settings.ticker):
    print(f"  [{h['score']:.3f}] ({h['doc_type']:10}) {h['text'][:70]}...")

# Correctness: filing query returns a filing
filing_hits = wiki.search("SEC 10-K 10-Q annual quarterly report", ticker=settings.ticker,
                           k=1, doc_type="filing")
assert filing_hits and filing_hits[0]["doc_type"] == "filing", "expected filing hit"
print(f"\nFiling URL: {filing_hits[0]['source_url']}")
print("Retrieval verified — all doc types indexed and retrievable.")

── search_each_type (guaranteed doc-type coverage):
  [0.770] (segment   ) NVDA Revenue by Market Platform (2026-04-26): Data Center Revenue $75....
  [0.757] (geo       ) NVDA Revenue by Geography (2026-04-26): United States revenue $63.77B ...
  [0.713] (segment   ) NVDA Revenue by Segment (2026-04-26): Compute and Networking revenue $...
  [0.541] (transcript) We delivered another record quarter while navigating what continues to...
  [0.383] (financial ) NVDA FY2026Q2 income statement (2026-04-30): Total revenue $81.61B (+8...
  [0.270] (transcript) Yes. Thank you for the question. I want to go back to the Spectrum XGS...
  [0.000] (news      ) Why SSR Mining Stock Got Knocked Today — Gold and silver are eternally...
  [0.000] (news      ) Will This Jensen Huang-Endorsed AI Chip Stock Be the Next Trillion-Dol...
  [-0.039] (filing    ) NVDA SEC 8-K — Current Report (Material Events) filed 2025-08-27 (peri...
  [-0.039] (filing    ) NVDA SEC 8-K — Current Report (Material Events) fi

### Switching to real embeddings
Set `EMBEDDING_BACKEND=sentence-transformers` (local `bge-base-en-v1.5`) or `vllm` (MI300X
`/v1/embeddings`). The BM25 alpha auto-adjusts to 0.35 / 0.15 respectively — better dense
vectors need less keyword correction.

**Next (Step 3):** market sentiment from real news (`03_market_sentiment.ipynb`).